# Student LLM on Colab: train, serve, generate 2,000 buildings

Distillation step 3 of the pipeline: fine-tune **Qwen2.5-7B-Instruct** with LoRA on the teacher's
verified answers, serve it with **vLLM**, and let it generate **2,000** labelled buildings. Every
answer goes through the same rule-engine check as the teacher's (`src/llm/generate.py`).

**Before you start**
1. *Runtime > Change runtime type*: **L4** or **A100** GPU (a T4 cannot serve the 7B model in 16-bit).
2. A GitHub fine-grained token with *read* access to `Byomer1021/ada-llm-gnn`, saved as a Colab
   secret named **`GH_TOKEN`** (key icon on the left; turn on notebook access).
3. Upload `D:\ada-llm-gnn\data\llm\student_sft.jsonl` to Google Drive:
   **`MyDrive/ada-llm-gnn/data/llm/student_sft.jsonl`**.

Every output (adapter, records, IFC files, metrics) is written to `MyDrive/ada-llm-gnn/`, so a
disconnect loses nothing; the generation cell continues with a new seed. Keep the tab open.

*Status: written and checked offline only. The training script ran end to end on CPU with a tiny
model; this notebook has not run on a GPU yet, so the first run may need a fix.*

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM: {VRAM_GB:.0f} GiB")
# vLLM serves the 7B model in 16-bit (about 15 GiB of weights + the KV cache)
assert VRAM_GB >= 22, "Runtime > Change runtime type: pick an L4 or an A100 GPU" 

In [ ]:
import os
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ada-llm-gnn"  # every output lives here
for d in ("data/llm", "data/synthetic", "models", "experiments"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)

REPO = "/content/ada-llm-gnn"
if not os.path.exists(REPO):
    url = f"https://{userdata.get('GH_TOKEN')}@github.com/Byomer1021/ada-llm-gnn.git"
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    if r.returncode:  # never print the URL: it contains the token
        raise RuntimeError("git clone failed: check the GH_TOKEN secret (read access to the repo)")
os.chdir(REPO)
for d in ("data", "models", "experiments"):  # the repo's output folders -> Drive
    if not os.path.islink(d):
        subprocess.run(["rm", "-rf", d], check=True)
        os.symlink(f"{DRIVE}/{d}", d)

SFT = "data/llm/student_sft.jsonl"
assert os.path.exists(SFT), f"upload student_sft.jsonl to {DRIVE}/data/llm/ first"
print(sum(1 for _ in open(SFT)), "training examples;", subprocess.run(
    ["git", "log", "--format=%h %s", "-1"], capture_output=True, text=True).stdout)

## 1. Train the student (LoRA)

Settings: `config/student.yaml`. A100 (40 / 80 GB): 16-bit LoRA; L4 (24 GB): 4-bit QLoRA.
Loss only on the teacher's answer; examples over 10,240 tokens are dropped (3 of 231).
Out of memory? Add `--max-length 8192` (drops 19 examples; do not go lower, the long
reasoning traces would be lost first). Metrics: `experiments/<time>_student/metrics.json`.

In [ ]:
!pip install -q -r requirements-student.txt

In [ ]:
FOUR_BIT = VRAM_GB < 40
flags = "" if FOUR_BIT else "--no-4bit"
print("4-bit QLoRA" if FOUR_BIT else "16-bit LoRA")
!python -m src.llm.train_student {flags}

In [ ]:
import glob

ADAPTER = os.path.realpath(sorted(glob.glob("models/student/*/adapter"))[-1])
print("adapter:", ADAPTER)

## 2. Serve the base model + the adapter with vLLM

Installed after training: vLLM brings its own torch / transformers versions. The first start
downloads the base model (~15 GB) and can take several minutes. Log: `/content/vllm.log`.

In [ ]:
!pip install -q vllm ifcopenshell openai pyyaml

In [ ]:
import json
import time
import urllib.request

import yaml

lora_r = yaml.safe_load(open("config/student.yaml"))["lora"]["r"]
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct", "--enable-lora", "--max-lora-rank", str(lora_r),
     "--lora-modules", f"qwen-ada={ADAPTER}", "--max-model-len", "12288",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        models = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2))
        print("ready:", [m["id"] for m in models["data"]])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM stopped:\n" + open("/content/vllm.log").read()[-3000:])
        time.sleep(5)
else:
    raise TimeoutError("vLLM not ready after 20 minutes, see /content/vllm.log")

## 3. Pilot: 20 buildings

Look at the pass rate before the big run. The records go to `data/llm/student_raw.jsonl`
(every attempt, with its problems) and `student_verified.jsonl`; they count toward the 2,000.

In [ ]:
!python -m src.llm.generate --role student --provider local --model qwen-ada --n 20 --seed 1000 --workers 8 --max-tokens 10000 --write-ifc

## 4. (E1) Untrained Qwen, 50 buildings

The same prompts with the base model, no adapter, in their own files
(`data/llm/student_base_*.jsonl`): how much did the fine-tuning add? Optional.

In [ ]:
!python -m src.llm.generate --role student --tag student_base --provider local --model Qwen/Qwen2.5-7B-Instruct --n 50 --seed 1 --workers 8 --max-tokens 10000

## 5. Generate until 2,000 buildings pass

Runs in chunks; each chunk takes the next unused seed, so running this cell again after a
disconnect continues where it stopped. It stops by itself if a chunk's pass rate falls below 20%.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, REPO)
from src.llm.generate import used_seeds

TARGET, CHUNK, MIN_RATE = 2000, 400, 0.2
VER, RAW = Path("data/llm/student_verified.jsonl"), Path("data/llm/student_raw.jsonl")
count = lambda p: sum(1 for _ in open(p)) if p.exists() else 0
while count(VER) < TARGET:
    seed = max(used_seeds(RAW) | {999}) + 1  # never reuse a seed, also after a disconnect
    n = min(CHUNK, int((TARGET - count(VER)) * 1.25) + 5)
    v0, r0 = count(VER), count(RAW)
    get_ipython().system(f"python -m src.llm.generate --role student --provider local --model qwen-ada "
                         f"--n {n} --seed {seed} --workers 16 --max-tokens 10000 --write-ifc")
    rate = (count(VER) - v0) / max(count(RAW) - r0, 1)
    print(f"seed {seed}: kept {count(VER) - v0}/{count(RAW) - r0} ({rate:.0%}), verified in total {count(VER)}")
    if rate < MIN_RATE:
        print("Pass rate too low: stopping. Look at the problems in data/llm/student_raw.jsonl.")
        break

In [ ]:
from collections import Counter

recs = [json.loads(line) for line in open(RAW)]
kept = [r for r in recs if r["kept"]]
print(f"attempts {len(recs)}, kept {len(kept)} ({len(kept) / len(recs):.0%})")
print("labels:", Counter(r["labels"]["verdict"]["label"] for r in kept))
print("scenarios:", dict(Counter(r["scenario"] for r in kept)))
print("top problems:", Counter(p.split(":")[0][:70] for r in recs for p in r["problems"]).most_common(8))
tok = [r["usage"]["completion_tokens"] for r in recs if r.get("usage")]
print(f"answer tokens: mean {sum(tok) / len(tok):.0f}, max {max(tok)}")

## 6. Bring the results home

Everything is in `MyDrive/ada-llm-gnn/`. Copy into `D:\ada-llm-gnn\`:
`data/llm/student_*.jsonl`, `data/synthetic/student/`, `models/student/`, `experiments/*_student/`.
Or download one zip with the cell below. Then, on the PC:
`python -m src.agents.dataset` and retrain the three agents.

In [ ]:
!cd {DRIVE} && zip -qr /content/student_outputs.zip data/llm/student_raw.jsonl data/llm/student_verified.jsonl data/llm/student_base_raw.jsonl data/llm/student_base_verified.jsonl data/synthetic/student models/student experiments
from google.colab import files

files.download("/content/student_outputs.zip")

In [ ]:
server.terminate()  # stop vLLM